In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, glob, shutil, hashlib, subprocess, time, math
from pathlib import Path

DRIVE_ROOT   = Path('/content/drive/MyDrive')
PARENT_DIR   = DRIVE_ROOT / 'CALSHIFT_Research'
PROJECT_ROOT = PARENT_DIR / 'calshift-research'
CRED_DIR     = DRIVE_ROOT / '.gitcreds'

subprocess.run(['git','config','--global','user.name','Md Anas Biswas'], check=False)
subprocess.run(['git','config','--global','user.email','anasbiswas@gmail.com'], check=False)
subprocess.run(['git','config','--global','credential.helper','store'], check=False)

for fn, dest in [('.git-credentials','/root/.git-credentials'),
                 ('.gitconfig','/root/.gitconfig')]:
    for cand in (PARENT_DIR / fn, CRED_DIR / fn):
        if cand.exists():
            shutil.copy(cand, dest); os.chmod(dest, 0o600); break

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
subprocess.run(['git','pull','--ff-only','--quiet'], check=False)

import importlib
if 'config' in sys.modules: importlib.reload(sys.modules['config'])
import config
import numpy as np, pandas as pd
print('ready:', os.getcwd())

Mounted at /content/drive
ready: /content/drive/MyDrive/CALSHIFT_Research/calshift-research


In [ ]:
# =============================================================================
# 08_mechanism
#
# EXPLORATORY. Not specified in the preregistration. Per preregistration
# section 16, every analysis in this notebook is labelled exploratory in the
# paper and none of it is used to evaluate any gate or criterion.
#
# Question: WHY does source-held-out calibration lose coverage? The gates
# establish THAT it does. This establishes the mechanism.
# =============================================================================
EXPLORATORY = True
print('EXPLORATORY ANALYSIS - no gate or criterion is evaluated here')

nsl_train = pd.read_parquet(config.INTERIM_DIR / 'nslkdd_train.parquet').reset_index(drop=True)
nsl_test  = pd.read_parquet(config.INTERIM_DIR / 'nslkdd_test.parquet').reset_index(drop=True)
part = pd.read_parquet(config.PROC_DIR / 'nslkdd_source_partition_labels.parquet')
nsl_train = nsl_train.assign(partition=part['partition'].values)

CLASSES = config.CANONICAL_CLASSES; K = len(CLASSES)
c2i = {c: i for i, c in enumerate(CLASSES)}
FOCAL = json.loads((config.REPORTS_DIR / 'focal_class_record.json').read_text())['focal_class']
FOCAL_I = c2i[FOCAL]
ALPHA = config.ALPHA_PRIMARY

S_pool = nsl_train[nsl_train.partition == 'source_cal_pool']
y_sp = S_pool['label'].map(c2i).to_numpy()
y_te = nsl_test['label'].map(c2i).to_numpy()
seen = set(nsl_train['subtype']); Spool_sub = set(S_pool['subtype'])

assign = pd.read_parquet(config.PROC_DIR / 'nslkdd_ladder_assignments.parquet')
IDX = {(r, j, role): g['test_idx'].to_numpy()
       for (r, j, role), g in assign.groupby(['rung','realization','role'])}
MODELS = sorted(glob.glob(str(config.PROC_DIR / 'probs_*.npz')))
print(f'{len(MODELS)} models | focal {FOCAL} | alpha {ALPHA}')

EXPLORATORY ANALYSIS - no gate or criterion is evaluated here


In [ ]:
# =============================================================================
# Cell 3 - scores (identical spec to notebooks 05 and 07)
# =============================================================================
from numpy.random import Generator, Philox

def draw_U(n, k, seed, stream):
    return Generator(Philox(key=int(seed), counter=int(stream))).random((n, k))

def aps_scores(P, U):
    gt = P[:, None, :] > P[:, :, None]
    return np.einsum('iyj,ij->iy', gt.astype(P.dtype), P) + U * P

def conformal_q(scores, alpha):
    n = len(scores)
    if n == 0: return np.inf
    k = math.ceil((n + 1) * (1 - alpha))
    return np.inf if k > n else float(np.sort(scores)[k - 1])

print('helpers ready')

In [ ]:
# =============================================================================
# Cell 4 - where does the source quantile sit relative to target scores?
# For each class: the source calibration quantile, and the fraction of target
# true-label scores that exceed it. That fraction IS the coverage loss.
# =============================================================================
rows = []
for mfile in MODELS:
    arch, seed = Path(mfile).stem.replace('probs_', '').rsplit('_s', 1); seed = int(seed)
    z = np.load(mfile)
    P_te = z['target'].astype(np.float64); P_sp = z['S_pool'].astype(np.float64)
    S_te = aps_scores(P_te, draw_U(len(P_te), K, seed, 1))
    S_sp = aps_scores(P_sp, draw_U(len(P_sp), K, seed, 2))

    for c, cname in enumerate(CLASSES):
        cal = S_sp[y_sp == c, c]
        if len(cal) == 0: continue
        q = conformal_q(cal, ALPHA)
        tgt = S_te[y_te == c, c]
        if len(tgt) == 0: continue
        rows.append(dict(arch=arch, seed=seed, cls=cname,
                         n_cal=len(cal), q_source=q,
                         cal_median=float(np.median(cal)),
                         tgt_median=float(np.median(tgt)),
                         frac_tgt_above_q=float((tgt > q).mean()),
                         tgt_p01=float(np.quantile(tgt, 0.01)),
                         tgt_frac_near1=float((tgt > 0.999).mean())))

mech = pd.DataFrame(rows)
agg = mech.groupby('cls')[['n_cal','q_source','cal_median','tgt_median',
                           'frac_tgt_above_q','tgt_frac_near1']].mean().round(4)
print('source quantile versus target score distribution, alpha =', ALPHA)
print(agg.to_string())
print('\nfrac_tgt_above_q is the coverage LOSS: 1 - coverage under SHC')

In [ ]:
# =============================================================================
# Cell 5 - the exclusion mechanism
# An APS score near 1.0 for the true label means the model assigned that label
# almost no probability, so every other class sits above it. Test directly:
# what probability does the model give the true class on target points?
# =============================================================================
rows = []
for mfile in MODELS:
    arch, seed = Path(mfile).stem.replace('probs_', '').rsplit('_s', 1); seed = int(seed)
    P_te = np.load(mfile)['target'].astype(np.float64)
    p_true = P_te[np.arange(len(y_te)), y_te]
    pred = P_te.argmax(1)
    for c, cname in enumerate(CLASSES):
        m = y_te == c
        if not m.any(): continue
        rows.append(dict(arch=arch, seed=seed, cls=cname, n=int(m.sum()),
                         p_true_median=float(np.median(p_true[m])),
                         frac_p_true_lt_001=float((p_true[m] < 0.01).mean()),
                         frac_p_true_lt_1e6=float((p_true[m] < 1e-6).mean()),
                         accuracy=float((pred[m] == c).mean()),
                         frac_confident_wrong=float(((pred[m] != c) &
                                                     (P_te[m].max(1) > 0.9)).mean())))
exc = pd.DataFrame(rows)
print('what probability does the model give the TRUE class on target data?')
print(exc.groupby('cls')[['p_true_median','frac_p_true_lt_001','frac_p_true_lt_1e6',
                          'accuracy','frac_confident_wrong']].mean().round(4).to_string())
print('\nfrac_confident_wrong: predicted a different class with p > 0.9')

In [ ]:
# =============================================================================
# Cell 6 - subtype composition: what was the source quantile calibrated ON?
# =============================================================================
src_sub = (S_pool[S_pool.label == FOCAL].groupby('subtype').size()
           .sort_values(ascending=False).rename('n_source'))
tgt_sub = (nsl_test[nsl_test.label == FOCAL].groupby('subtype').size()
           .sort_values(ascending=False).rename('n_target'))
comp = pd.concat([src_sub, tgt_sub], axis=1).fillna(0).astype(int)
comp['in_source'] = comp['n_source'] > 0
comp['pct_source'] = (comp['n_source'] / comp['n_source'].sum() * 100).round(1)
comp['pct_target'] = (comp['n_target'] / comp['n_target'].sum() * 100).round(1)

print(f'{FOCAL} subtype composition, source calibration pool versus target')
print(comp.sort_values('n_target', ascending=False).to_string())
share = comp['pct_source'].max()
top = comp['pct_source'].idxmax()
tgt_unseen_mass = comp.loc[~comp.in_source, 'n_target'].sum() / comp['n_target'].sum()
print(f'\nsource calibration is {share:.1f}% "{top}"')
print(f'{tgt_unseen_mass:.1%} of target {FOCAL} is in subtypes ABSENT from the source pool')

In [ ]:
# =============================================================================
# Cell 7 - per-subtype coverage under SHC
# Which target subtypes actually fail, and does source representation predict it?
# =============================================================================
rows = []
for mfile in MODELS[:10]:            # 10 models is ample for a descriptive breakdown
    arch, seed = Path(mfile).stem.replace('probs_', '').rsplit('_s', 1); seed = int(seed)
    z = np.load(mfile)
    P_te = z['target'].astype(np.float64); P_sp = z['S_pool'].astype(np.float64)
    S_te = aps_scores(P_te, draw_U(len(P_te), K, seed, 1))
    S_sp = aps_scores(P_sp, draw_U(len(P_sp), K, seed, 2))
    q = np.array([conformal_q(S_sp[y_sp == c, c], ALPHA) for c in range(K)])
    covered = (S_te <= q[None, :])[np.arange(len(y_te)), y_te]
    for st, g in nsl_test[nsl_test.label == FOCAL].groupby('subtype'):
        i = g.index.to_numpy()
        rows.append(dict(subtype=st, n=len(i), coverage=float(covered[i].mean()),
                         n_source=int(src_sub.get(st, 0))))

sub = (pd.DataFrame(rows).groupby(['subtype','n','n_source'])['coverage']
       .mean().reset_index().sort_values('n', ascending=False))
sub['in_source'] = sub['n_source'] > 0
print(f'SHC coverage by target {FOCAL} subtype')
print(sub.round(4).to_string(index=False))
print('\nmean coverage, subtypes present in source :',
      round(float(sub[sub.in_source]['coverage'].mean()), 4))
print('mean coverage, subtypes absent from source:',
      round(float(sub[~sub.in_source]['coverage'].mean()), 4))

In [ ]:
# =============================================================================
# Cell 8 - why the gap shrinks at strict alpha
# At alpha 0.01 the source quantile sits closer to 1.0, so it still admits
# points the model got confidently wrong. Quantify that directly.
# =============================================================================
rows = []
for mfile in MODELS[:10]:
    arch, seed = Path(mfile).stem.replace('probs_', '').rsplit('_s', 1); seed = int(seed)
    z = np.load(mfile)
    P_te = z['target'].astype(np.float64); P_sp = z['S_pool'].astype(np.float64)
    S_te = aps_scores(P_te, draw_U(len(P_te), K, seed, 1))
    S_sp = aps_scores(P_sp, draw_U(len(P_sp), K, seed, 2))
    cal = S_sp[y_sp == FOCAL_I, FOCAL_I]; tgt = S_te[y_te == FOCAL_I, FOCAL_I]
    for a in [0.01, 0.05, 0.10, 0.20]:
        q = conformal_q(cal, a)
        m = y_te == FOCAL_I
        size = (S_te[m] <= np.array([conformal_q(S_sp[y_sp == c, c], a)
                                     for c in range(K)])[None, :]).sum(1)
        rows.append(dict(alpha=a, q_source=q, coverage=float((tgt <= q).mean()),
                         gap_to_one=float(1.0 - q),
                         set_size=float(size.mean()),
                         frac_full_set=float((size == K).mean())))
al = pd.DataFrame(rows).groupby('alpha')[['q_source','coverage','gap_to_one',
                                          'set_size','frac_full_set']].mean()
al['nominal'] = 1 - al.index
al['shortfall'] = al['nominal'] - al['coverage']
print(f'{FOCAL}: source quantile position and resulting coverage by alpha')
print(al.round(4).to_string())
print('\nas alpha tightens the quantile approaches 1.0 and admits more of the')
print('confidently-wrong mass, so the shortfall shrinks -- but read set_size:')
print('coverage recovered at q = 1.0 is VACUOUS, the set is every label.')
vac = al[(al['q_source'] >= 0.9999)]
if len(vac):
    print(f'  VACUOUS at alpha in {list(vac.index)}: set size '
          f'{vac["set_size"].round(2).tolist()} of {K}, '
          f'full-set rate {vac["frac_full_set"].round(3).tolist()}')

In [ ]:
# =============================================================================
# Cell 9 - figures
# =============================================================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
config.FIGURES_DIR.mkdir(parents=True, exist_ok=True)

z = np.load(MODELS[0]); seed = int(Path(MODELS[0]).stem.rsplit('_s', 1)[1])
P_te = z['target'].astype(np.float64); P_sp = z['S_pool'].astype(np.float64)
S_te = aps_scores(P_te, draw_U(len(P_te), K, seed, 1))
S_sp = aps_scores(P_sp, draw_U(len(P_sp), K, seed, 2))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
cal = S_sp[y_sp == FOCAL_I, FOCAL_I]; tgt = S_te[y_te == FOCAL_I, FOCAL_I]
q = conformal_q(cal, ALPHA)
axes[0].hist(cal, bins=40, alpha=0.65, density=True, label='source calibration')
axes[0].hist(tgt, bins=40, alpha=0.65, density=True, label='target evaluation')
axes[0].axvline(q, color='k', ls='--', label=f'source q ({q:.3f})')
axes[0].set_xlabel('APS score for the true label'); axes[0].set_ylabel('density')
axes[0].set_title(f'{FOCAL}: score distributions'); axes[0].legend(fontsize=8)

s2 = sub.sort_values('coverage')
cols = ['tab:red' if not v else 'tab:blue' for v in s2['in_source']]
axes[1].barh(s2['subtype'], s2['coverage'], color=cols)
axes[1].axvline(1 - ALPHA, color='k', ls='--', lw=1)
axes[1].set_xlabel('SHC coverage'); axes[1].set_title(f'{FOCAL} coverage by subtype')
axes[1].tick_params(labelsize=7)
plt.tight_layout()
plt.savefig(config.FIGURES_DIR / 'mechanism_nslkdd.png', dpi=160)
print('saved figures/mechanism_nslkdd.png')
plt.close()

In [ ]:
# =============================================================================
# Cell 10 - persist and commit
# =============================================================================
mech.to_csv(config.REPORTS_DIR / 'mechanism_quantile_position.csv', index=False)
exc.to_csv(config.REPORTS_DIR / 'mechanism_true_class_probability.csv', index=False)
comp.to_csv(config.REPORTS_DIR / 'mechanism_subtype_composition.csv')
sub.to_csv(config.REPORTS_DIR / 'mechanism_subtype_coverage.csv', index=False)
al.to_csv(config.REPORTS_DIR / 'mechanism_alpha_dependence.csv')

note = {'status': 'EXPLORATORY, not preregistered',
        'focal_class': FOCAL, 'alpha': ALPHA,
        'source_calibration_concentration': float(comp['pct_source'].max()),
        'dominant_source_subtype': str(comp['pct_source'].idxmax()),
        'target_mass_absent_from_source': float(tgt_unseen_mass),
        'coverage_subtypes_in_source': float(sub[sub.in_source]['coverage'].mean()),
        'coverage_subtypes_absent': float(sub[~sub.in_source]['coverage'].mean())}
(config.REPORTS_DIR / 'mechanism_note.json').write_text(json.dumps(note, indent=2))
print(json.dumps(note, indent=2))

def git(*a, show=True):
    r = subprocess.run(['git', *a], capture_output=True, text=True)
    if show:
        if r.stdout.strip(): print(r.stdout.strip())
        if r.stderr.strip(): print(r.stderr.strip())
    return r
for s, dd in [('/root/.git-credentials', PARENT_DIR / '.git-credentials'),
              ('/root/.gitconfig',       PARENT_DIR / '.gitconfig')]:
    if os.path.exists(s): shutil.copy(s, dd)
os.chdir(PROJECT_ROOT); git('add','-A', show=False)
if git('status','--porcelain', show=False).stdout.strip():
    git('commit','-m','nb08: exploratory mechanism diagnostics')
    rr = git('push','-u','origin','main')
    if rr.returncode: print('PUSH FAILED. Commit is safe locally.')
else:
    print('nothing to commit')
print(git('log','--oneline','-3', show=False).stdout)